In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("silver_data").getOrCreate()
df_spark = spark.read.parquet("../../data/silver/btc_silver/part-00000-71101873-a382-4526-8a7e-92fa23e57435-c000.snappy.parquet")
df_spark.show(5)
df_spark.count()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/20 23:35:32 WARN Utils: Your hostname, DESKTOP-NSV5698, resolves to a loopback address: 127.0.1.1; using 172.25.203.92 instead (on interface eth0)
26/01/20 23:35:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/20 23:35:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+
|          open_time|    open|    high|     low|   close|  volume|          close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|
+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+
|2026-01-20 05:26:00|91664.58|91675.01|91651.94|91658.54|10.09312|2026-01-20 05:26:...|    925148.4640845|            3015|              3.96286|        363220.3147199|     0|
|2026-01-20 05:27:00|91658.54|91747.05|91658.54|91713.95| 17.8326|2026-01-20 05:27:...|   1635452.6459378|            3905|              9.31005|        853705.3018537|     0|
|2026-01-20 05:28:00|91713.96|91731.39|91696.42|91696.42|   8.581|2026-01-20 05:28:...|    787002.9017224|            23

600

In [2]:
from pyspark.sql import Window
import pyspark.sql.functions as F

w_time = Window.orderBy("open_time")
# Create target variable: close price at t+10
df_feat = df_spark.withColumn("close_t_plus_10",F.lead("close", 10).over(w_time))
df_feat.show(5)

26/01/20 23:36:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+
|          open_time|    open|    high|     low|   close|  volume|          close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|close_t_plus_10|
+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+
|2026-01-20 05:26:00|91664.58|91675.01|91651.94|91658.54|10.09312|2026-01-20 05:26:...|    925148.4640845|            3015|              3.96286|        363220.3147199|     0|       91730.59|
|2026-01-20 05:27:00|91658.54|91747.05|91658.54|91713.95| 17.8326|2026-01-20 05:27:...|   1635452.6459378|            3905|              9.31005|        853705.3018537|     0|       91762.59|
|2026-01-20 05:28:00|91713.96|91731.39|9

In [3]:
# 4) Return (variation relative)
df_feat = df_feat.withColumn("close_prev",F.lag("close", 1).over(w_time))
df_feat = df_feat.withColumn("return_1m",(F.col("close") - F.col("close_prev")) / F.col("close_prev"))
df_feat.show(5)

26/01/20 23:36:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+----------+--------------------+
|          open_time|    open|    high|     low|   close|  volume|          close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|close_t_plus_10|close_prev|           return_1m|
+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+----------+--------------------+
|2026-01-20 05:26:00|91664.58|91675.01|91651.94|91658.54|10.09312|2026-01-20 05:26:...|    925148.4640845|            3015|              3.96286|        363220.3147199|     0|       91730.59|      NULL|                NULL|
|2026-01-20 05:27:00|91658.54|91747.05|91658.54|91713.95| 17.8326|2026-01-20 05:27:...|   1635452.645937

In [4]:
# 5) Moyennes mobiles 5 et 10 minutes
w_5 = Window.orderBy("open_time").rowsBetween(-4, 0)
w_10 = Window.orderBy("open_time").rowsBetween(-9, 0)

# 6) Moving averages
df_feat = df_feat.withColumn("ma_5",F.avg("close").over(w_5)).withColumn("ma_10",F.avg("close").over(w_10))
df_feat.show(5)
df_feat.printSchema()

26/01/20 23:36:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+----------+--------------------+-----------------+-----------------+
|          open_time|    open|    high|     low|   close|  volume|          close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|close_t_plus_10|close_prev|           return_1m|             ma_5|            ma_10|
+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+----------+--------------------+-----------------+-----------------+
|2026-01-20 05:26:00|91664.58|91675.01|91651.94|91658.54|10.09312|2026-01-20 05:26:...|    925148.4640845|            3015|              3.96286|        363220.3147199|     0|       91730.59|      NULL|                NU

In [5]:
# 6) Taker ratio
df_feat = df_feat.withColumn("taker_ratio",F.col("taker_buy_base_volume") / F.col("volume"))
df_feat.show(5)

26/01/20 23:36:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+----------+--------------------+-----------------+-----------------+-------------------+
|          open_time|    open|    high|     low|   close|  volume|          close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|close_t_plus_10|close_prev|           return_1m|             ma_5|            ma_10|        taker_ratio|
+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+----------+--------------------+-----------------+-----------------+-------------------+
|2026-01-20 05:26:00|91664.58|91675.01|91651.94|91658.54|10.09312|2026-01-20 05:26:...|    925148.4640845|            3015|              3.96286|        363220.

In [6]:
from pyspark.sql.functions import col, sum

df_feat.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_feat.columns
]).show(truncate=False)


26/01/20 23:36:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------+----+----+---+-----+------+----------+------------------+----------------+---------------------+----------------------+------+---------------+----------+---------+----+-----+-----------+
|open_time|open|high|low|close|volume|close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|close_t_plus_10|close_prev|return_1m|ma_5|ma_10|taker_ratio|
+---------+----+----+---+-----+------+----------+------------------+----------------+---------------------+----------------------+------+---------------+----------+---------+----+-----+-----------+
|0        |0   |0   |0  |0    |0     |0         |0                 |0               |0                    |0                     |0     |10             |1         |1        |0   |0    |0          |
+---------+----+----+---+-----+------+----------+------------------+----------------+---------------------+----------------------+------+---------------+----------+---------+----+-----+-----------+



In [7]:
# 7) supprimant toutes les lignes où une feature ou la cible est manquante.
df_feat = df_feat.dropna(subset=["close_t_plus_10", "return_1m", "close_prev"])
df_feat.show(5)

26/01/20 23:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+----------+--------------------+-----------------+-----------------+-------------------+
|          open_time|    open|    high|     low|   close|  volume|          close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|close_t_plus_10|close_prev|           return_1m|             ma_5|            ma_10|        taker_ratio|
+-------------------+--------+--------+--------+--------+--------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+----------+--------------------+-----------------+-----------------+-------------------+
|2026-01-20 05:27:00|91658.54|91747.05|91658.54|91713.95| 17.8326|2026-01-20 05:27:...|   1635452.6459378|            3905|              9.31005|        853705.

In [8]:
df_feat.write.mode("overwrite").parquet("../../data/silver/btc_features/")
print("Table btc_features sauvegardée")

26/01/20 23:36:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/20 23:36:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Table btc_features sauvegardée
